In [ ]:
import numpy as np
import pandas as pd
from statsmodels.api import Logit 
import statsmodels.api as sm
import requests
import duckdb
import os
import s3fs
import sys
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep
%load_ext autoreload
%autoreload 2

np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"


FILE_KEY_S3 = "table_analyse_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df = pd.read_csv(file_in, sep=",")

df.head()   

In [ ]:
df = pd.read_pickle(
    "/home/onyxia/work/bistrots-et-bulletins/data_load_and_prep/table_analyse_muni.pkl"
)

In [ ]:
df.head()

# Préparation des données pour la régression 

In [ ]:
#df['POPULATION'].max()

In [ ]:
#Provisoire, le temps que la table soit enregistrée dans le bucket
df_sous_ech = df[df['POPULATION'] > 3500]

In [ ]:
df_sous_ech = df_sous_ech.drop(columns =['Unnamed: 0'])

In [ ]:
df_sous_ech.columns

In [ ]:
# Comoter le nombre de valeurs manquantes dans la table finale 

colonnes = [
    'POPULATION',
    'NIVVIE_MEDIAN',
    'TAUX_PAUVRETE',
    'nb_etablissements',
    'age_moyen_bistrots',
    'Nombre candidats',
    'Abstentions'
]

print(df_sous_ech[colonnes].isna().sum())


df_sous_ech[df_sous_ech[colonnes].isna().any(axis=1)]

In [ ]:
# Supprimer les valeurs manquantes : 
df_sous_ech = df_sous_ech.dropna(subset=['NIVVIE_MEDIAN']) # certaines communes ne sont pas diffusées dans filosofi par mesure de secret stat
df_sous_ech = df_sous_ech.dropna(subset=['TAUX_PAUVRETE']) # idem 
#df_sous_ech = df_sous_ech.dropna(subset=['age_moyen_bistrots'])  on garde car c'est pour les villes n'ayant pas de bistrot

In [ ]:
# Préparation des variables 
df_sous_ech['ind1']= df_sous_ech['ind1'].astype("category")
df_sous_ech['ind2']= df_sous_ech['ind2'].astype("category")
df_sous_ech['ind3']= df_sous_ech['ind3'].astype("category")
df_sous_ech['ind4']= df_sous_ech['ind4'].astype("category")
df_sous_ech["bistrot_dens"]=df_sous_ech['nb_etablissements']/df_sous_ech['POPULATION']*1000
df_sous_ech["tx_abs"] = df_sous_ech['Abstentions']/(df_sous_ech['Inscrits'])

In [ ]:
df_sous_ech.describe()

In [ ]:
## Matrice de variance covariance pour reprérer les corrélations 
col = ['POPULATION', 'NIVVIE_MEDIAN', 'TAUX_PAUVRETE', 'bistrot_dens', 'age_moyen_bistrots','Nombre candidats','tx_abs']

# Autre table parce que normalisation à venir
df_cor = df_sous_ech

## Normaliser les valeurs 
df_cor[ col ] = (df_sous_ech[col] - df_sous_ech[col].mean()) / df_sous_ech[col].std()

## Matrice de corrélation
cor_matrix = df_cor[col].corr(method='pearson')

display(cor_matrix.round(2)) #pas de grosse corrélation : max entre nb candidats et taux de pauvreté. Faire 2 régressions, une avec, une sans. 


## Déterminants du vote de droite 

### Régression logistique simple

In [ ]:
# Choix des variables
X = df_sous_ech[['bistrot_dens']]
y = df_sous_ech['ind1']

## Ajouter une constante
X = sm.add_constant(X)

## Fit OLS model
lr = Logit(endog=y,exog=X) 
res = lr.fit() 

#résumé des résultats 
print(res.summary()) 

# Coefficient positif : un bistrot de plus est associé à une probabilité plus élevée que la liste élue soit de droite
# Ce n'est pas le sens attendu : il faut contrôler avec d'autres variables. 

### Régression linéaire multiple

In [ ]:
# Régression linéaire multiple 



## Déterminants du vote RN